# Step 3 · Outflow of activity: where did residents of counties without power go?

Step 2 found that in counties with 72+ hours of outage in a week, visits to local places fell by about a quarter. This step uses GeoDS county-to-county flows to answer three questions:

| Question | Approach |
| --- | --- |
| **Q1 Moved or stopped?** Of the visits residents no longer made in their own county, how many turned up in other counties? | Residents' visits are split into "home county" and "other counties" and compared with counties without outages in the same event and week, giving "how much home visits fell" and "how much visits elsewhere rose"; their ratio is the **displacement share**, with an interval from resampling counties |
| **Q2 Where to?** What kind of counties received the extra visits? | Each hit county's usual split across other counties is the baseline; in the outage week, which destinations got more than their usual share: by distance, the destination's own power, its population and whether it is in the same state (Poisson regression, PPML) |
| **Q3 Who absorbed them, and does that inflate Step 2?** | For each county, the share of its usual visitors who come from counties currently in a 72+ h outage, called **network exposure**. Do highly exposed counties get more visits? Then exposure is added to the Step 2c model to see whether the 72+ h coefficient changes |

**Why Q3 matters**: Step 2's comparison group is "counties with under 6 hours of outage that week". If some of them took in visitors from the hit counties, their visits rose, the gap between hit counties and the comparison group widened, and Step 2 overstated the effect.

**Data**: Step 1b's `step1b_edges_<event>.csv.gz` (county-pair flows over each event's 12 weeks, for pairs starting or ending in the study area) and `geods_weeks/week_totals.csv` (national weekly visits, used to express flows per million visits nationwide); the county boundaries downloaded in Step 2c (for distances between counties; downloaded again if missing); county population from Opportunity Insights. The sample is the same as in Step 2c: 7 events, without Uri.

**Structure**: A data → B the decision rules, fixed in advance → C flows → D Q1 → E Q2 → F Q3 → G map → H checklist and conclusions. No Earth Engine is needed; it runs in a few minutes.

In [ ]:
# 1. Environment, folders, chart style
import os, sys, glob, zipfile, subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from IPython.display import display
import statsmodels.api as sm
import statsmodels.formula.api as smf
try:
    import geopandas as gpd
except ImportError:                     # Colab usually has geopandas; install it if not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)
    import geopandas as gpd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz", "step2a_hazard_weekly.csv",
          "geods_weeks/week_totals.csv", "oi_geoids_county.csv"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run Steps 1b and 2a first.")

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
HOLDOUT = ["uri2021"]                          # as in Step 2, left out
WEEK_FROM, WEEK_TO = -5, 6
BASE_COLS = [-5, -4, -3, -2]                   # baseline weeks, as in Step 1b
BAD_WEEK_PCT = 1.0                             # baseline weeks with a mean outage share of 1% or more are not used (as in Step 1b)
MIN_BASE_WEEKS = 2
HIT_HOURS = 72                                 # a "hit county-week": 72+ hours without power per customer that week
DIST_EDGES = [0, 50, 150, 400, np.inf]         # distance between county centroids (km)
DIST_CODES = ["d0", "d1", "d2", "d3"]
POP_EDGES = [0, 100_000, 500_000, np.inf]      # destination population
POP_CODES = ["s_small", "s_mid", "s_large"]
EXPO_EDGES = [-np.inf, 0.005, 0.02, np.inf]    # network exposure bands: < 0.5%, 0.5–2%, ≥ 2%
EXPO_CODES = ["e0", "e1", "e2"]
MIN_LIFT = 10                                  # Q2: a change must be at least 10% to count as a direction (tiny but significant differences do not)
B_BOOT = 999
SEED = 2026
Z = norm.ppf(0.975)
LABEL = {"d0": "0–50 km", "d1": "50–150 km", "d2": "150–400 km", "d3": "400+ km", "dna": "distance unknown",
         "p_none": "had power (< 6 h)", "p_some": "partial outage (6–72 h)", "p_long": "also without power (72+ h)", "p_out": "outside the study area",
         "s_small": "< 100k people", "s_mid": "100k–500k people", "s_large": "500k+ people", "s_na": "population unknown",
         "e0": "< 0.5%", "e1": "0.5–2%", "e2": "≥ 2%"}
# Step 2 bands (for the Q3 controls)
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
GUST_EDGES = [-np.inf, 17.5, 25, 33, np.inf]
RAIN_EDGES = [-np.inf, 25, 75, 150, np.inf]
COLD_EDGES = [-np.inf, 5, 25, np.inf]
RAINCUM_EDGES = [-np.inf, 50, 150, 300, np.inf]
CUM_FROM = -1
TEST = "out4=h072p"

## A. Data

The same sample and variables as Step 2c (`MAIN`), plus:

- `HIT`: county-weeks in the main sample with 72+ hours of outage that week; the "hit counties" of Q1 and Q2
- `HIT_ANY`: every county-week in the study area with 72+ hours of outage that week (including counties left out of the main sample for baseline problems), used for Q3's network exposure, since residents of excluded counties travel too
- `CLEAN`: each county's usable baseline weeks (weeks −5 to −2 with a mean outage share under 1%)

In [ ]:
# 3. Sample (as in Step 2c), hit county-weeks, usable baseline weeks
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
for c in ["t0", "day0", "week0"]:
    EVENTS[c] = pd.to_datetime(EVENTS[c])
EV_MAIN = [k for k in EVENTS.index if k not in HOLDOUT]
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])
D = (W.merge(L[["event", "fips", "name", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left"))
D["out_bin"]     = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["out4"]        = D["out_bin"].replace({"h072": "h072p", "h120": "h072p"})
D["gust_bin"]    = pd.cut(D["gust_mean_ms"], GUST_EDGES, labels=["g0", "g1", "g2", "g3"]).astype(str)
D["gustcum_bin"] = pd.cut(D["gust_cum"], GUST_EDGES, labels=["k0", "k1", "k2", "k3"]).astype(str)
D["rain_bin"]    = pd.cut(D["rain_mm"], RAIN_EDGES, labels=["r0", "r1", "r2", "r3"]).astype(str)
D["raincum_bin"] = pd.cut(D["rain_cum"], RAINCUM_EDGES, labels=["s0", "s1", "s2", "s3"]).astype(str)
D["cold_bin"]    = pd.cut(D["cold_ppt_mm"], COLD_EDGES, labels=["c0", "c1", "c2"]).astype(str)
D["ew"] = D["event"] + "_" + D["rel_week"].astype(str)
D["pw"] = ["pre" if r <= -2 else ("4+" if r >= 4 else str(r)) for r in D["rel_week"]]
D["gw"] = np.where(D["gustcum_bin"] == "k0", "base", D["gustcum_bin"] + "|" + D["pw"])
D["rw"] = np.where(D["raincum_bin"] == "s0", "base", D["raincum_bin"] + "|" + D["pw"])
NEEDED = ["out_week_hours", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "gust_cum", "rain_cum"]
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D[NEEDED].notna().all(axis=1)
MAIN = D[D["keep"] & ~D["event"].isin(HOLDOUT)].copy()

HIT = MAIN.loc[MAIN["out4"] == "h072p", ["event", "fips", "name", "rel_week", "out_week_hours"]].reset_index(drop=True)
HIT_ANY = W.loc[(W["out_week_hours"] >= HIT_HOURS) & W["event"].isin(EV_MAIN), ["event", "fips", "rel_week"]]
CLEAN = W.loc[W["rel_week"].isin(BASE_COLS) & ~(W["out_week_mean_pct"] >= BAD_WEEK_PCT), ["event", "fips", "rel_week"]]
print(f"Main sample {len(MAIN):,} county-weeks; hit county-weeks (Q1, Q2): {len(HIT)} in {HIT['fips'].nunique()} counties; "
      f"all county-weeks with 72+ h in the study area (Q3): {len(HIT_ANY)}")
HIT.groupby("event").agg(county_weeks=("fips", "size"), counties=("fips", "nunique"), max_hours=("out_week_hours", "max"))

## B. The decision rules, fixed in advance

Written down before looking at any result.

**Q1 Moved or stopped** (displacement share = extra visits in other counties ÷ visits lost at home)

- ≥ 0.5: **most of it moved to other counties**
- 0.2–0.5: **part of it moved to other counties**
- < 0.2: **most of it did not happen** (activity stopped rather than moved)

The category follows the point estimate; if the 95% interval crosses 0.2 or 0.5, the result is "between A and B".

**Q2 Where to** (PPML, relative to each hit county's usual split across other counties; "significant" also requires a change of at least 10%, so tiny but statistically significant differences do not count)

- (a) **Avoided places also without power**: for destinations that also had 72+ h of outage that week, the coefficient is below 0 with an interval excluding 0 (vs destinations with power)
- (b) **Distance**: 150–400 km and 400+ km both significantly negative → "stayed nearby"; either significantly positive → "went farther than usual"; otherwise → "no clear change in distance" (both vs 0–50 km)
- (c) **Toward larger counties**: destinations with 500k+ people have a coefficient above 0 with an interval excluding 0 (vs under 100k)

**Q3 Spillovers and the Step 2 correction** (Step 2c's HxW model)

- (a) **Connected counties gained visits**: the coefficient of the ≥ 2% exposure band is above 0 with an interval excluding 0 (outcome `visits_in_chg`)
- (b) **Does Step 2 need correcting**: with exposure added, the 72+ h coefficient changes by less than 20% of its value and is still significantly negative → "Step 2 is not affected by spillovers"; otherwise → "Step 2 needs correcting", with the corrected value reported

## C. County locations and population; county-pair flows

- County centroids: the 2020 county boundaries downloaded in Step 2c (kept in `step2c_geo`), with centroids in an equal-area projection; the distance between two counties is the straight line between their centroids
- Population and county names: the Opportunity Insights county file saved in Step 1b
- Flows: each event's county-pair flows, divided by that week's national total of visits and multiplied by a million ("per million visits nationwide"), which removes national seasonal and pandemic swings. A pair missing in a week counts as 0

Each origin county's baseline is the mean over its usable baseline weeks (as in Step 1b). One reconciliation: residents' total visits added up from the county-pair flows should equal `visits_res` in the Step 1b panel exactly.

In [ ]:
# 4. County centroids, distances, population, names
GEO = f"{OUT}/step2c_geo"
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"

def county_shapes():
    # use the county boundaries downloaded in Step 2c; download them if missing
    path = f"{GEO}/county.zip"
    if not os.path.exists(path) or os.path.getsize(path) < 10_000:
        os.makedirs(GEO, exist_ok=True)
        r = requests.get(COUNTY_URL, timeout=600, headers={"User-Agent": "Mozilla/5.0 (research script)"})
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    folder = f"{GEO}/county"
    with zipfile.ZipFile(path) as z:
        z.extractall(folder)
    return gpd.read_file(glob.glob(f"{folder}/*.shp")[0])[["GEOID", "geometry"]].to_crs(5070)

SHAPES = county_shapes()
CENT = SHAPES.set_index("GEOID").geometry.centroid
CX, CY = CENT.x, CENT.y
OI = pd.read_csv(f"{OUT}/oi_geoids_county.csv")
OI["fips"] = OI["countyfips"].astype(int).astype(str).str.zfill(5)
POP = OI.set_index("fips")["county_pop2019"].astype(float)
NAMES = pd.Series((OI["countyname"] + ", " + OI["stateabbrev"]).to_numpy(), index=OI["fips"])

def band(x, edges, codes, missing):
    # put a continuous variable into bands; blanks become `missing`
    b = pd.cut(pd.Series(np.asarray(x, dtype=float)), edges, right=False, labels=codes).astype(object)
    return b.where(b.notna(), missing).astype(str).to_numpy()

def dist_km(o, d):
    # centroid distance between two lists of counties (km); counties without a boundary give blanks
    return np.hypot(CX.reindex(o).to_numpy() - CX.reindex(d).to_numpy(), CY.reindex(o).to_numpy() - CY.reindex(d).to_numpy()) / 1000

print(f"Boundaries for {len(SHAPES):,} counties; population for {POP.notna().sum():,} counties")

In [ ]:
# 5. Read county-pair flows: origins' home/other-county visits, pair flows of hit counties, "new destinations"
TOT = pd.read_csv(f"{OUT}/geods_weeks/week_totals.csv", parse_dates=["week"]).set_index("week")["total_visits"].astype(float)
WEEKS = list(range(WEEK_FROM, WEEK_TO + 1))
LOC, NON, NEW, RAW, PAIRS, CMASK = {}, {}, {}, [], {}, {}
for key in EV_MAIN:
    area = set(D.loc[D["event"] == key, "fips"])
    E = pd.read_csv(f"{OUT}/step1b_edges_{key}.csv.gz", dtype={"geoid_o": str, "geoid_d": str}, parse_dates=["week"])
    E = E[E["geoid_o"].isin(area)].copy()
    E["f"] = E["visitor_flows"] / E["week"].map(TOT).to_numpy() * 1e6          # per million visits nationwide
    home = E["geoid_o"] == E["geoid_d"]
    LOC[key] = E[home].pivot_table(index="geoid_o", columns="rel_week", values="f", aggfunc="sum").reindex(columns=WEEKS)
    NON[key] = E[~home].pivot_table(index="geoid_o", columns="rel_week", values="f", aggfunc="sum").reindex(columns=WEEKS)
    seen = E[E["rel_week"].isin(BASE_COLS)][["geoid_o", "geoid_d"]].drop_duplicates()
    nb = E[~home].merge(seen.assign(known=True), on=["geoid_o", "geoid_d"], how="left")
    NEW[key] = nb[nb["known"].isna()].pivot_table(index="geoid_o", columns="rel_week", values="f", aggfunc="sum").reindex(columns=WEEKS)
    RAW.append(E.groupby(["geoid_o", "rel_week"])["visitor_flows"].sum().rename("res_edges").reset_index().assign(event=key))
    origins = LOC[key].index.union(NON[key].index)
    LOC[key] = LOC[key].reindex(origins).fillna(0.0)
    NON[key] = NON[key].reindex(origins).fillna(0.0)
    NEW[key] = NEW[key].reindex(origins).fillna(0.0)
    cm = (CLEAN[CLEAN["event"] == key].assign(v=1.0).pivot_table(index="fips", columns="rel_week", values="v")
          .reindex(index=origins, columns=BASE_COLS).fillna(0.0))
    CMASK[key] = cm
    hit_o = set(HIT_ANY.loc[HIT_ANY["event"] == key, "fips"])
    P = (E[E["geoid_o"].isin(hit_o)].pivot_table(index=["geoid_o", "geoid_d"], columns="rel_week", values="f", aggfunc="sum")
         .reindex(columns=WEEKS).fillna(0.0))
    m = cm.reindex(P.index.get_level_values(0)).fillna(0.0).to_numpy()
    n = m.sum(axis=1)
    P["base_clean"] = np.where(n >= MIN_BASE_WEEKS, (P[BASE_COLS].to_numpy() * m).sum(axis=1) / np.maximum(n, 1), np.nan)
    P["base_plain"] = P[BASE_COLS].to_numpy().mean(axis=1)                       # weights for Q3: the plain mean of weeks −5 to −2
    PAIRS[key] = P
    print(f"{key}: {len(E):,} rows, {len(origins):,} origin counties, {len(P):,} pairs from hit counties")

RAW = pd.concat(RAW, ignore_index=True).rename(columns={"geoid_o": "fips"})
rec = RAW.merge(W[["event", "fips", "rel_week", "visits_res"]], on=["event", "fips", "rel_week"], how="inner")
rec = rec[rec["visits_res"] > 0]
RECONCILE = float((rec["res_edges"] / rec["visits_res"] - 1).abs().max())
print(f"Check: summed pair flows vs the panel's visits_res, largest relative difference {RECONCILE:.2e} ({len(rec):,} county-weeks)")

## D. Q1: moved or stopped

For each hit county-week (county o, week t):

- **Counterfactual**: in the same event and week, counties with no substantial outage over the whole event (`group == none`) and under 6 hours that week had home-county visits at r_L times their baseline and other-county visits at r_N times their baseline. Without the outage, o's home visits would have been r_L × baseline and its other-county visits r_N × baseline
- **Lost at home** = r_L × home baseline − actual home visits
- **Extra elsewhere** = actual other-county visits − r_N × other-county baseline
- **Displacement share** = "extra elsewhere" summed over all hit county-weeks ÷ "lost at home" summed the same way

Interval: 999 resamples of counties (all weeks of a county drawn together). Results are also given by outage duration and by event, and "extra elsewhere" is split by the destination's distance and by whether the destination had power that week.

In [ ]:
# 6. Q1: "lost at home" and "extra elsewhere" for each hit county-week
rows, cmp_n = [], []
for key in EV_MAIN:
    hits = HIT[HIT["event"] == key]
    if hits.empty:
        continue
    loc, non, cm = LOC[key], NON[key], CMASK[key]
    n = cm.sum(axis=1)
    Lb = ((loc[BASE_COLS] * cm).sum(axis=1) / n).where(n >= MIN_BASE_WEEKS)
    Nb = ((non[BASE_COLS] * cm).sum(axis=1) / n).where(n >= MIN_BASE_WEEKS)
    comp = MAIN[(MAIN["event"] == key) & (MAIN["group"] == "none") & (MAIN["out4"] == "h000")]
    for t, h in hits.groupby("rel_week"):
        cf = [f for f in comp.loc[comp["rel_week"] == t, "fips"] if f in Lb.index and pd.notna(Lb[f]) and Lb[f] > 0]
        rL = loc.loc[cf, t].sum() / Lb[cf].sum()
        rN = non.loc[cf, t].sum() / Nb[cf].sum()
        cmp_n.append(len(cf))
        for f, hours in zip(h["fips"], h["out_week_hours"]):
            if f not in Lb.index or pd.isna(Lb[f]):
                continue
            rows.append({"event": key, "fips": f, "rel_week": t, "hours": hours, "home_cf": rL * Lb[f], "lost": rL * Lb[f] - loc.loc[f, t],
                         "extra": non.loc[f, t] - rN * Nb[f], "rL": rL, "rN": rN, "new_now": NEW[key].loc[f, t], "non_now": non.loc[f, t]})
ACC = pd.DataFrame(rows)
ACC["name"] = ACC["fips"].map(NAMES)
ACC.to_csv(f"{OUT}/step3_accounting.csv", index=False)

def share_ci(a, boot=B_BOOT, seed=SEED):
    # displacement share and its 95% interval from resampling counties
    by = a.groupby("fips")[["extra", "lost"]].sum()
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(by), size=(boot, len(by)))
    ex, lo = by["extra"].to_numpy()[draws].sum(axis=1), by["lost"].to_numpy()[draws].sum(axis=1)
    return by["extra"].sum() / by["lost"].sum(), np.percentile(ex / lo, 2.5), np.percentile(ex / lo, 97.5)

SHARE, SHARE_LO, SHARE_HI = share_ci(ACC)
home_pct = -ACC["lost"].sum() / ACC["home_cf"].sum() * 100
else_pct = ACC["extra"].sum() / ACC["home_cf"].sum() * 100
print(f"{len(ACC)} hit county-weeks (at least {min(cmp_n)} comparison counties per event-week)")
print(f"Home-county visits: {home_pct:+.0f}% (vs the counterfactual); visits elsewhere: {else_pct:+.0f}% (also as a share of the home counterfactual)")
print(f"Displacement share: {SHARE:.2f}, 95% interval [{SHARE_LO:.2f}, {SHARE_HI:.2f}]")

def q1_cat(x):
    return "most" if x >= 0.5 else ("part" if x >= 0.2 else "stopped")
Q1_TEXT = {"most": "most of it moved to other counties", "part": "part of it moved to other counties",
           "stopped": "most of it did not happen (it stopped)"}
lo_c, pt_c, hi_c = q1_cat(SHARE_LO), q1_cat(SHARE), q1_cat(SHARE_HI)
Q1_VERDICT = Q1_TEXT[pt_c] if lo_c == hi_c else f"between \"{Q1_TEXT[lo_c]}\" and \"{Q1_TEXT[hi_c]}\" (point estimate: {Q1_TEXT[pt_c]})"
print("Q1 conclusion:", Q1_VERDICT)

parts = []
for label, a in [("72–120 h", ACC[ACC["hours"] < 120]), ("120+ h", ACC[ACC["hours"] >= 120])] + [(k, ACC[ACC["event"] == k]) for k in ACC["event"].unique()]:
    if len(a) == 0:
        continue
    s, lo_, hi_ = share_ci(a)
    parts.append({"group": label, "county_weeks": len(a), "home_change_pct": -a["lost"].sum() / a["home_cf"].sum() * 100,
                  "elsewhere_change_pct": a["extra"].sum() / a["home_cf"].sum() * 100, "share": s, "lo": lo_, "hi": hi_})
tidy(pd.DataFrame(parts).set_index("group"), 2)

In [ ]:
# 7. Split "extra elsewhere" by destination county
pieces = []
for key in ACC["event"].unique():
    P = PAIRS[key]
    outw = W[W["event"] == key].set_index(["fips", "rel_week"])["out_week_hours"]
    for _, r in ACC[ACC["event"] == key].iterrows():
        p = P.loc[r["fips"]]
        p = p[p.index != r["fips"]]
        pieces.append(pd.DataFrame({"event": key, "o": r["fips"], "rel_week": r["rel_week"], "d": p.index.to_numpy(),
                                    "now": p[r["rel_week"]].to_numpy(), "base": p["base_clean"].to_numpy(), "rN": r["rN"],
                                    "dest_hours": outw.reindex(list(zip(p.index, [r["rel_week"]] * len(p)))).to_numpy()}))
PX = pd.concat(pieces, ignore_index=True)
PX["excess"] = PX["now"] - PX["rN"] * PX["base"]
PX["km"] = dist_km(PX["o"], PX["d"])
PX["dist"] = band(PX["km"], DIST_EDGES, DIST_CODES, "dna")
PX["power"] = np.select([PX["dest_hours"].isna(), PX["dest_hours"] >= HIT_HOURS, PX["dest_hours"] >= 6], ["p_out", "p_long", "p_some"], "p_none")
PX["pop"] = POP.reindex(PX["d"]).to_numpy()
PX["size"] = band(PX["pop"], POP_EDGES, POP_CODES, "s_na")
PX["same_state"] = (PX["o"].str[:2] == PX["d"].str[:2]).astype(float)
gap = (PX.groupby(["event", "o", "rel_week"])["excess"].sum().reset_index()
       .merge(ACC[["event", "fips", "rel_week", "extra"]], left_on=["event", "o", "rel_week"], right_on=["event", "fips", "rel_week"]))
print("Split check: destinations summed vs extra elsewhere, largest difference", f"{(gap['excess'] - gap['extra']).abs().max():.2e}")

home_cf = ACC["home_cf"].sum()
DEST = pd.concat([
    pd.DataFrame({"split": "distance", "code": ["home"], "pct": [-ACC["lost"].sum() / home_cf * 100]}),
    PX.groupby("dist")["excess"].sum().div(home_cf).mul(100).rename("pct").reset_index().rename(columns={"dist": "code"}).assign(split="distance"),
    PX.groupby("power")["excess"].sum().div(home_cf).mul(100).rename("pct").reset_index().rename(columns={"power": "code"}).assign(split="power"),
], ignore_index=True)
DEST["label"] = DEST["code"].map(LABEL).fillna("home county")
DEST.to_csv(f"{OUT}/step3_destinations.csv", index=False)
new_hit = ACC["new_now"].sum() / ACC["non_now"].sum()
print(f"In the outage week, share of other-county visits going to counties never visited in the baseline: {new_hit:.1%}")
tidy(DEST[["split", "label", "pct"]], 1)

In [ ]:
# 8. Figure 1: lost at home and extra elsewhere
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), sharex=True)
specs = [("distance", ["home"] + DIST_CODES, "By distance from the hit county"),
         ("power", ["p_none", "p_some", "p_long", "p_out"], "Extra visits elsewhere, by the destination's power that week")]
for ax, (split, order, title) in zip(axes, specs):
    t = DEST[DEST["split"] == split].set_index("code").reindex(order).dropna(subset=["pct"])
    labels = ["Home county" if c == "home" else LABEL[c] for c in t.index]
    colors = [C_GREY if c == "home" else (C_DARK if v >= 0 else C_LIGHT) for c, v in zip(t.index, t["pct"])]
    y = np.arange(len(t))[::-1]
    ax.barh(y, t["pct"], color=colors, height=0.6)
    for yi, v in zip(y, t["pct"]):
        inside = v < -15                                                      # long negative bars: the number goes inside the bar
        ax.text(v + (0.8 if v >= 0 else (1.0 if inside else -0.8)), yi, f"{v:+.1f}%", va="center", ha="left" if (v >= 0 or inside) else "right",
                fontsize=9, color=SURFACE if inside else INK)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.axvline(0, color=FAINT, lw=1)
    ax.grid(axis="y", visible=False)
    ax.set_title(title)
    ax.set_xlabel("Residents' visits vs the counterfactual,\n% of the visits they would have made in their home county")
lim = max(10, DEST["pct"].abs().max() * 1.25)
axes[0].set_xlim(-lim, lim)
fig.suptitle(f"Where the activity went: {len(ACC)} county-weeks with 72+ h of outage, displacement share {SHARE:.2f} [{SHARE_LO:.2f}, {SHARE_HI:.2f}]",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step3_fig1_accounting.png"); plt.show()

## E. Q2: where to

Poisson regression (PPML). Each row is "hit county-week × one destination county", using only destinations visited in the baseline (baseline flow > 0):

- Outcome: the flow that week
- Offset: log(r_N × baseline flow), i.e. "what this destination would get at the usual split"
- Fixed effects: one per hit county-week, absorbing "how much this county's travel changed that week"
- Explanatory variables: distance (vs 0–50 km), the destination's power that week (vs had power), the destination's population (vs under 100k), same state

Coefficients are turned into percentages, exp(coefficient) − 1: "how much more (or less) than usual this kind of destination received, compared with the reference". Standard errors are clustered by hit county.

In [ ]:
# 9. Q2: PPML
Q = PX[(PX["base"] > 0) & (PX["dist"] != "dna") & (PX["size"] != "s_na")].copy()
Q["offset"] = np.log(Q["rN"] * Q["base"])
Q["ow"] = Q["event"] + "_" + Q["o"] + "_" + Q["rel_week"].astype(str)
TERMS = {"dist": "d0", "power": "p_none", "size": "s_small"}
formula = "now ~ " + " + ".join(f"C({v}, Treatment('{r}'))" for v, r in TERMS.items()) + " + same_state + C(ow)"
ppml = smf.glm(formula, data=Q, family=sm.families.Poisson(), offset=Q["offset"]).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(Q["o"])[0]})
ci = ppml.conf_int()
rows = []
for p in ppml.params.index:
    if p.startswith("C(ow)") or p == "Intercept":
        continue
    code_ = "same_state" if p == "same_state" else p.split("[T.")[1].rstrip("]")
    rows.append({"code": code_, "label": "same state" if code_ == "same_state" else LABEL[code_], "coef": ppml.params[p], "lo": ci.loc[p, 0], "hi": ci.loc[p, 1],
                 "pct": (np.exp(ppml.params[p]) - 1) * 100, "pct_lo": (np.exp(ci.loc[p, 0]) - 1) * 100, "pct_hi": (np.exp(ci.loc[p, 1]) - 1) * 100})
LIFT = pd.DataFrame(rows).set_index("code")
LIFT.to_csv(f"{OUT}/step3_ppml.csv")
print(f"{len(Q):,} hit county-week × destination rows; {Q['ow'].nunique()} hit county-weeks; {len(PX) - len(Q):,} rows dropped for a zero baseline or missing distance/population")

def sig(code_, sign):
    if code_ not in LIFT.index:
        return None
    r = LIFT.loc[code_]
    return bool(r["hi"] < 0 and r["pct"] <= -MIN_LIFT) if sign < 0 else bool(r["lo"] > 0 and r["pct"] >= MIN_LIFT)

Q2A = sig("p_long", -1)
far_neg = [sig("d2", -1), sig("d3", -1)]
far_pos = [sig("d2", +1), sig("d3", +1)]
Q2B = "went farther than usual" if any(x for x in far_pos if x) else (
      "stayed nearby" if all(x for x in far_neg if x is not None) and None not in far_neg else "no clear change in distance")
Q2C = sig("s_large", +1)
print("Q2 (a) avoided places also without power:", Q2A)
print("Q2 (b) distance:", Q2B)
print("Q2 (c) toward larger counties:", Q2C)
tidy(LIFT[["label", "pct", "pct_lo", "pct_hi"]], 0)

In [ ]:
# 10. Figure 2: which destinations got more than usual
ORDER = [("Distance (vs 0–50 km)", ["d1", "d2", "d3"]), ("Destination's power that week (vs had power)", ["p_some", "p_long", "p_out"]),
         ("Destination population (vs under 100k)", ["s_mid", "s_large"]), ("Same state (vs another state)", ["same_state"])]
rows_ = [(g, [c for c in cs if c in LIFT.index]) for g, cs in ORDER]
rows_ = [(g, cs) for g, cs in rows_ if cs]
pos, heads, ticks, labs = 0.0, [], [], []
for g, cs in rows_:                                                       # each group gets a header row, then its points
    heads.append((pos, g))
    pos -= 1
    for c in cs:
        ticks.append(pos)
        labs.append(c)
        pos -= 1
    pos -= 0.4
fig, ax = plt.subplots(figsize=(9, 0.42 * (len(ticks) + len(heads)) + 1.6))
for yi, c in zip(ticks, labs):
    r = LIFT.loc[c]
    ax.errorbar(r["pct"], yi, xerr=[[r["pct"] - r["pct_lo"]], [r["pct_hi"] - r["pct"]]], fmt="o", color=C_DARK, ms=6, elinewidth=1.6, capsize=0)
ax.set_yticks(ticks)
ax.set_yticklabels([LIFT.loc[c, "label"] for c in labs])
for yh, g in heads:
    ax.text(-0.01, yh, g, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=9, fontweight="bold", color=INK)
ax.set_ylim(pos + 0.6, 0.6)
ax.axvline(0, color=FAINT, lw=1)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Visits from the hit county, % more (or less) than its usual split would give (95% interval)")
ax.set_title(f"Which destinations received the displaced visits ({Q['ow'].nunique()} hit county-weeks, PPML)")
fig.tight_layout()
fig.savefig(f"{OUT}/step3_fig2_destinations.png"); plt.show()

## F. Q3: network exposure, spillovers and the Step 2 correction

County d's **network exposure** in week t = Σ (usual flow to d from counties o with 72+ h of outage that week) ÷ d's usual total visits. The weights come from the baseline (weeks −5 to −2) and outage status from EAGLE-I; neither depends on d's visits that week, so exposure can serve as an explanatory variable.

Exposure bands (vs < 0.5%) are then added to Step 2c's HxW model (event × week fixed effects, outage bands, this week's wind and rain and week-varying cumulative wind and rain):

- Outcomes `visits_in_chg` (used by the decision rule) and `visits_in_nonlocal_chg` (visitors from other counties, the most direct sign of spillover)
- An exclusion check too: non-hit county-weeks with exposure ≥ 2% are dropped and the 72+ h coefficient re-estimated

In [ ]:
# 11. Network exposure
clean_in = W.merge(CLEAN, on=["event", "fips", "rel_week"])
BASE_IN = clean_in.groupby(["event", "fips"])["visits_in_pm"].mean()
ex_rows = []
for key in EV_MAIN:
    P = PAIRS[key]
    o_lvl, d_lvl = P.index.get_level_values(0), P.index.get_level_values(1)
    for t, g in HIT_ANY[HIT_ANY["event"] == key].groupby("rel_week"):
        sel = o_lvl.isin(set(g["fips"])) & (o_lvl != d_lvl)
        inflow = P.loc[sel, "base_plain"].groupby(level=1).sum()
        base_in = BASE_IN.reindex(list(zip([key] * len(inflow), inflow.index))).to_numpy()
        ex_rows.append(pd.DataFrame({"event": key, "fips": inflow.index.to_numpy(), "rel_week": t, "expo": inflow.to_numpy() / base_in}))
EXPO = pd.concat(ex_rows, ignore_index=True).dropna()
MAIN = MAIN.drop(columns=["expo"], errors="ignore").merge(EXPO, on=["event", "fips", "rel_week"], how="left")
MAIN["expo"] = MAIN["expo"].fillna(0.0)
MAIN["expo_band"] = band(MAIN["expo"], EXPO_EDGES, EXPO_CODES, "e0")
print("County-weeks per exposure band in the main sample (hit county-weeks excluded):",
      MAIN.loc[MAIN["out4"] != "h072p", "expo_band"].value_counts().reindex(EXPO_CODES).to_dict())
top = (MAIN[MAIN["out4"] != "h072p"].nlargest(10, "expo")[["event", "fips", "name", "rel_week", "expo", "out_week_hours", "visits_in_chg"]]
       .assign(expo=lambda d: (d["expo"] * 100).round(1)))
top

In [ ]:
# 12. Q3: models with exposure
OUT_X = [("out4", "h000")]
HXW_X = [("gust_bin", "g0"), ("rain_bin", "r0"), ("cold_bin", "c0"), ("gw", "base"), ("rw", "base")]
EXPO_X = [("expo_band", "e0")]
WEBB = np.array([-np.sqrt(1.5), -1.0, -np.sqrt(0.5), np.sqrt(0.5), 1.0, np.sqrt(1.5)])

def dummies(d, specs):
    # banded variables → 0/1 columns (reference band dropped), named "variable=band"
    parts = [pd.DataFrame(index=d.index)]
    for col, ref in specs:
        dm = pd.get_dummies(d[col].astype(str), prefix=col, prefix_sep="=", dtype=float)
        parts.append(dm.drop(columns=[f"{col}={ref}"], errors="ignore"))
    return pd.concat(parts, axis=1)

def fe_ols(data, y, xspec, fe="ew", test=TEST, boot=0, seed=SEED):
    # as in Step 2c: least squares after removing event × week fixed effects, clustered by county (CR1); optional wild cluster bootstrap
    d = data.dropna(subset=[y])
    X = dummies(d, xspec)
    X = X.loc[:, X.sum() > 0]
    codes = pd.factorize(d[fe])[0]
    S = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))))
    M = np.column_stack([d[y].to_numpy(float), X.to_numpy()])
    M = M - ((S @ M) / np.bincount(codes)[:, None])[codes]
    yv, Xm = M[:, 0], M[:, 1:]
    live = np.linalg.norm(Xm, axis=0) > 1e-7 * np.sqrt(len(yv))
    names, Xm = list(X.columns[live]), Xm[:, live]
    N = len(yv)
    XtX_inv = np.linalg.pinv(Xm.T @ Xm)
    A = XtX_inv @ Xm.T
    b = A @ yv
    u = yv - Xm @ b
    g = pd.factorize(d["fips"])[0]
    G = int(g.max()) + 1
    C = sp.csr_matrix((np.ones(N), (g, np.arange(N))), shape=(G, N))
    K = int(np.linalg.matrix_rank(Xm)) + d[fe].nunique()
    corr = G / (G - 1) * (N - 1) / (N - K)
    meat = C @ (Xm * u[:, None])
    se = np.sqrt(np.clip(np.diag(corr * XtX_inv @ (meat.T @ meat) @ XtX_inv), 0, None))
    table = pd.DataFrame({"coef": b, "se": se, "lo": b - Z * se, "hi": b + Z * se, "p_boot": np.nan}, index=names)
    if boot and test in names:
        k = names.index(test)
        Xr = np.delete(Xm, k, axis=1)
        fit_r = Xr @ np.linalg.lstsq(Xr, yv, rcond=None)[0]
        u_r = yv - fit_r
        t_obs = b[k] / se[k]
        rng = np.random.default_rng(seed)
        t_star = []
        for start in range(0, boot, 100):
            v = rng.choice(WEBB, size=(G, min(100, boot - start)))
            Ys = fit_r[:, None] + u_r[:, None] * v[g]
            Bs = A @ Ys
            s = C @ (A[k][:, None] * (Ys - Xm @ Bs))
            t_star.append(Bs[k] / np.sqrt(corr * (np.asarray(s) ** 2).sum(axis=0)))
        table.loc[test, "p_boot"] = float(np.mean(np.abs(np.concatenate(t_star)) >= abs(t_obs)))
    return table, N

MODELS = []
def keep_rows(label, outcome, table, n, data):
    for term in [TEST, "expo_band=e1", "expo_band=e2"]:
        if term in table.index:
            r = table.loc[term]
            MODELS.append({"model": label, "outcome": outcome, "term": term, "coef": r["coef"], "lo": r["lo"], "hi": r["hi"], "p_boot": r["p_boot"],
                           "n_obs": n, "n_cells": int((data[term.split("=")[0]].astype(str) == term.split("=")[1]).sum())})

for y in ["visits_in_chg", "visits_in_nonlocal_chg"]:
    t0, n0 = fe_ols(MAIN, y, OUT_X + HXW_X, boot=B_BOOT)
    keep_rows("M0 Step 2c (HxW)", y, t0, n0, MAIN)
    for test in [TEST, "expo_band=e2"]:
        t1, n1 = fe_ols(MAIN, y, OUT_X + HXW_X + EXPO_X, test=test, boot=B_BOOT)
        if test == TEST:
            keep_rows("M1 + exposure", y, t1.drop(index=["expo_band=e1", "expo_band=e2"], errors="ignore"), n1, MAIN)
        else:
            keep_rows("M1 + exposure", y, t1.loc[[c for c in t1.index if c.startswith("expo_band=")]], n1, MAIN)
excl = MAIN[~((MAIN["expo_band"] == "e2") & (MAIN["out4"] != "h072p"))]
t2, n2 = fe_ols(excl, "visits_in_chg", OUT_X + HXW_X, boot=B_BOOT)
keep_rows("M2 exposed counties dropped", "visits_in_chg", t2, n2, excl)
MOD = pd.DataFrame(MODELS)
MOD.to_csv(f"{OUT}/step3_exposure_models.csv", index=False)

def pick_row(model, outcome, term):
    s = MOD[(MOD["model"] == model) & (MOD["outcome"] == outcome) & (MOD["term"] == term)]
    return s.iloc[0] if len(s) else None

e2 = pick_row("M1 + exposure", "visits_in_chg", "expo_band=e2")
Q3A = None if e2 is None else bool(e2["lo"] > 0)
b0, b1 = pick_row("M0 Step 2c (HxW)", "visits_in_chg", TEST), pick_row("M1 + exposure", "visits_in_chg", TEST)
CHANGE = (b1["coef"] - b0["coef"]) / abs(b0["coef"])
Q3B = bool(abs(CHANGE) < 0.2 and b1["hi"] < 0)
print("Q3 (a) connected counties gained visits:", Q3A)
print(f"Q3 (b) 72+ h coefficient: {b0['coef']:+.1f} → {b1['coef']:+.1f} (change {CHANGE:+.0%}); "
      + ("Step 2 is not affected by spillovers" if Q3B else "Step 2 needs correcting"))
show = MOD.assign(cell=MOD.apply(lambda r: f"{r['coef']:+.1f} [{r['lo']:+.1f}, {r['hi']:+.1f}] n={r['n_cells']}", axis=1))
show.pivot_table(index=["outcome", "term"], columns="model", values="cell", aggfunc="first")

In [ ]:
# 13. Figure 3: spillovers and the Step 2 coefficient
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.3), gridspec_kw={"width_ratios": [1.1, 1]})
for i, (y, color, label) in enumerate([("visits_in_chg", C_DARK, "All visits to the county"), ("visits_in_nonlocal_chg", C_MID, "Visitors from other counties")]):
    xs, cs, los, his = [], [], [], []
    for j, code_ in enumerate(EXPO_CODES):
        r = pick_row("M1 + exposure", y, f"expo_band={code_}")
        if code_ == "e0":
            xs.append(j); cs.append(0.0); los.append(0.0); his.append(0.0)
        elif r is not None:
            xs.append(j); cs.append(r["coef"]); los.append(r["lo"]); his.append(r["hi"])
    xs = np.array(xs) + (i - 0.5) * 0.18
    ax1.errorbar(xs, cs, yerr=[np.array(cs) - np.array(los), np.array(his) - np.array(cs)], fmt="o", color=color, ms=6, elinewidth=1.6, capsize=0, label=label)
n_band = MAIN.loc[MAIN["out4"] != "h072p", "expo_band"].value_counts()
ax1.set_xticks(range(len(EXPO_CODES)))
ax1.set_xticklabels([f"{LABEL[c]}\n(n = {int(n_band.get(c, 0))})" for c in EXPO_CODES])
ax1.axhline(0, color=FAINT, lw=1)
ax1.set_xlabel("Share of the county's usual visitors who come from counties with 72+ h of outage that week")
ax1.set_ylabel("Log points vs exposure under 0.5%")
ax1.set_title("Spillover onto connected counties (95% intervals)")
ax1.legend(fontsize=8.5, loc="upper left")
mods = [("M0 Step 2c (HxW)", "Step 2c model"), ("M1 + exposure", "+ network exposure"), ("M2 exposed counties dropped", "Exposed counties dropped")]
for yi, (m, lab) in zip([2, 1, 0], mods):
    r = pick_row(m, "visits_in_chg", TEST)
    if r is not None:
        ax2.errorbar(r["coef"], yi, xerr=[[r["coef"] - r["lo"]], [r["hi"] - r["coef"]]], fmt="o", color=C_DARK, ms=6, elinewidth=1.6, capsize=0)
        ax2.text(r["coef"], yi + 0.18, f"{r['coef']:+.1f}", ha="center", va="bottom", fontsize=9, color=INK)
ax2.set_yticks([2, 1, 0])
ax2.set_yticklabels([lab for _, lab in mods])
ax2.set_ylim(-0.6, 2.7)
ax2.axvline(0, color=FAINT, lw=1)
ax2.grid(axis="y", visible=False)
ax2.set_xlabel("72+ h coefficient, log points (95% interval)")
ax2.set_title("Does the spillover change the Step 2 estimate?")
fig.tight_layout()
fig.savefig(f"{OUT}/step3_fig3_spillover.png"); plt.show()

## G. Map: where residents went in the week with the most outages

The event and week with the most hit county-weeks are chosen (week 1 after Ida), and each destination county is colored by the "extra visits" it received from the hit counties, as a percentage of all extra visits (log color scale). Black marks the hit counties themselves; gray marks counties that received no extra visits. The map covers the 12 largest receivers and the hit counties; only numbers appear on the map, with the top 8 named on the right.

In [ ]:
# 14. Figure 4: map
key_m, t_m = HIT.groupby(["event", "rel_week"]).size().idxmax()
sub = PX[(PX["event"] == key_m) & (PX["rel_week"] == t_m)]
recv = sub.groupby("d")["excess"].sum()
pos = recv[recv > 0]
share = (pos / pos.sum() * 100).sort_values(ascending=False)
hit_m = set(HIT.loc[(HIT["event"] == key_m) & (HIT["rel_week"] == t_m), "fips"])
top = [f for f in share.index if f not in hit_m][:12]
xmin, ymin, xmax, ymax = SHAPES[SHAPES["GEOID"].isin(set(top) | hit_m)].total_bounds
pad = 0.08 * max(xmax - xmin, ymax - ymin)
view = SHAPES.cx[xmin - pad:xmax + pad, ymin - pad:ymax + pad].copy()
view["share"] = view["GEOID"].map(share)
cmap = LinearSegmentedColormap.from_list("blues", ["#dbe8f9", C_LIGHT, C_MID, C_DARK])
aspect = (ymax - ymin + 2 * pad) / (xmax - xmin + 2 * pad)
fig, (ax, side) = plt.subplots(1, 2, figsize=(13, float(np.clip(9.8 * aspect + 1.0, 5.5, 10))), gridspec_kw={"width_ratios": [4, 1.3]})
view.plot(ax=ax, color="#eeede9", edgecolor="white", linewidth=0.3)
got = view[view["share"].notna() & ~view["GEOID"].isin(hit_m)]
vmin = max(0.05, float(got["share"].min())) if len(got) else 0.05
norm_ = LogNorm(vmin=vmin, vmax=max(float(share.max()), vmin * 1.01))
got.plot(ax=ax, column="share", cmap=cmap, norm=norm_, edgecolor="white", linewidth=0.3)
view[view["GEOID"].isin(hit_m)].plot(ax=ax, color=INK, edgecolor="white", linewidth=0.3)
cent = view.set_index("GEOID").geometry.centroid
listed = [f for f in top if f in cent.index][:8]
for i, f in enumerate(listed, 1):                                        # only numbers on the map, names on the right, to avoid overlaps
    ax.annotate(str(i), (cent[f].x, cent[f].y), ha="center", va="center", fontsize=8, fontweight="bold", color=INK,
                bbox=dict(boxstyle="circle,pad=0.25", facecolor=SURFACE, edgecolor=INK, linewidth=0.6))
ax.set_axis_off()
ax.set_title(f"{EVENTS.loc[key_m, 'name']}, week {t_m}: where residents of the {len(hit_m)} hit counties (black) went", loc="left")
side.set_axis_off()
cax = side.inset_axes([0.05, 0.58, 0.1, 0.38])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm_), cax=cax)
cb.set_label("Share of the extra visits\nfrom the hit counties (%)", fontsize=9)
side.text(0.0, 0.5, "Largest receivers", fontsize=10, fontweight="bold", color=INK, transform=side.transAxes, va="top")
for i, f in enumerate(listed, 1):
    side.text(0.0, 0.5 - 0.05 * i, f"{i}  {NAMES.get(f, f)}  {share[f]:.1f}%", fontsize=9, color=INK, transform=side.transAxes, va="top")
fig.tight_layout()
fig.savefig(f"{OUT}/step3_fig4_map.png"); plt.show()
tidy(pd.DataFrame({"county": [NAMES.get(f, f) for f in share.index[:10]], "share_pct": share.iloc[:10].to_numpy(),
                   "km_to_nearest_hit": [np.nanmin(dist_km([f] * len(hit_m), sorted(hit_m))) for f in share.index[:10]]}), 1)

In [ ]:
# 15. Checklist and conclusions
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

checks = [
    ("Summed pair flows = panel visits_res (largest relative difference)", f"{RECONCILE:.1e}", "< 0.01", RECONCILE < 0.01),
    ("Share of hit county-weeks with a usable baseline", round(len(ACC) / len(HIT), 3), "≥ 0.9", len(ACC) / len(HIT) >= 0.9),
    ("Comparison counties per event-week (minimum)", int(min(cmp_n)), "≥ 30", min(cmp_n) >= 30),
    ("Destinations add up to \"extra elsewhere\" (largest difference)", f"{(gap['excess'] - gap['extra']).abs().max():.1e}", "< 1e-6",
     bool((gap["excess"] - gap["extra"]).abs().max() < 1e-6)),
    ("PPML converged", str(ppml.converged), "True", bool(ppml.converged)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step3_checklist.csv", index=False)
display(check_tab)

def yes_no(x):
    return "—" if x is None else ("yes" if x else "no")
conclusions = pd.DataFrame([
    ("Q1", f"displacement share {SHARE:.2f} [{SHARE_LO:.2f}, {SHARE_HI:.2f}]; home {home_pct:+.0f}%, elsewhere {else_pct:+.0f}%", Q1_VERDICT),
    ("Q2a", "avoided places also without power", yes_no(Q2A)),
    ("Q2b", "distance", Q2B),
    ("Q2c", "toward larger counties", yes_no(Q2C)),
    ("Q3a", "connected counties gained visits (exposure ≥ 2%)", yes_no(Q3A)),
    ("Q3b", f"72+ h coefficient {b0['coef']:+.1f} → {b1['coef']:+.1f} ({CHANGE:+.0%})",
     "Step 2 is not affected by spillovers" if Q3B else "Step 2 needs correcting"),
], columns=["Question", "Value", "Conclusion"])
conclusions.to_csv(f"{OUT}/step3_conclusions.csv", index=False)
conclusions